# BigBang-Proton — matbench_mp_e_form Benchmark

**Paper:** Wu, Liu, He, Xin, Wu et al. *BigBang-Proton Technical Report: Next-Word-Prediction is Scientific Multitask Learner*, arXiv:2510.00129 (2025). https://arxiv.org/abs/2510.00129

**Resources:**
- Code: https://github.com/supersymmetry-technologies/BigBang-Proton
- Model: https://huggingface.co/SuperSymmetryTechnologies/BigBang-Proton

---

## Algorithm

BigBang-Proton is a 1.5B-parameter unified sequence-based architecture for auto-regressive language modeling, pretrained **from scratch** on cross-scale, cross-structure, cross-discipline real-world scientific tasks to construct a scientific multi-task learner. Three fundamental innovations:

1. **Theory-Experiment Learning**: aligns large-scale numerical experimental data with theoretical text corpora;
2. **Binary Patch Encoding**: replaces BPE tokenization, transforming textual/numerical/symbolic scientific data into byte sequences (259-symbol vocabulary);
3. **Monte Carlo Attention**: substitutes traditional transformer attention.

The base model is pretrained end-to-end by next-word prediction on multidisciplinary datasets — quark jet tagging (particle physics), inter-atomic potential simulations (materials science), genome and protein sequence structure prediction, lake water quality (spatio-temporal sensor) prediction, and up-to-50-digit arithmetic operations — mixed with general textual corpus (SlimPajama).

**Adaptation to matbench_mp_e_form** (formation energy per atom, eV/atom):

1. Supervised fine-tuning (SFT) of the base model on the MPTRJ (Materials Project + JARVIS) datasets -> BigBang-Proton-MPTRJ;
2. Each crystal structure is serialized into a compact deterministic text representation ending with the fixed prompt *"The formation energy per atom of the lattice is:"*;
3. The model autoregressively generates the formation-energy value as a byte string (e.g. `-1.23456 eV.`), parsed back to float;
4. Standard matbench protocol: one model fine-tuned per fold on the official train/val split, starting from BigBang-Proton-MPTRJ weights;
5. Test-time decoding is greedy (argmax) with temperature-1e-4 fallback sampling for non-digit tokens; per fold, best-loss and best-acc checkpoints are averaged.

**Results (official test folds, MAE eV/atom):** fold_0 0.0914, fold_1 0.0896, fold_2 0.0887, fold_3 0.0934, fold_4 0.0871; 5-fold mean 0.0900.

## Recording the benchmark results

The complete per-fold predictions are provided in `results.json.gz` (this package). The cell below re-records them through the official `MatbenchBenchmark` flow (`get_test_data` -> `record` -> `to_file`), exactly as described in the matbench documentation. This reproduces the submitted `results.json.gz` from the saved prediction arrays.

In [ ]:
import gzip
import json
import numpy as np
from matbench.bench import MatbenchBenchmark

# Saved per-fold prediction arrays (test.jsonl order == official test order)
PRED_FILES = {
    0: "preds_fold0.npy",
    1: "preds_fold1.npy",
    2: "preds_fold2.npy",
    3: "preds_fold3.npy",
    4: "preds_fold4.npy",
}

mb = MatbenchBenchmark(autoload=False, subset=["matbench_mp_e_form"])
task = list(mb.tasks)[0]
task.load()  # downloads the official dataset on first run

for fold in task.folds:
    test_inputs = task.get_test_data(fold, include_target=False)
    preds = np.load(PRED_FILES[fold]).astype(float)
    assert len(preds) == len(test_inputs), f"fold{fold}: length mismatch"
    task.record(fold, preds)  # predictions are floats (eV/atom)
    print(f"fold{fold}: recorded {len(preds)} predictions")

mb.to_file("results.json.gz")
print("saved results.json.gz")

## Verification

Reload the submission file through the official loader path to confirm it validates (5 folds, 132,752 predictions).

In [ ]:
with gzip.open("results.json.gz", "rt", encoding="utf-8") as f:
    d = json.load(f)
mb2 = MatbenchBenchmark.from_dict(d)  # official load path
for t in mb2.tasks:
    for f in t.folds:
        n = len(t.predictions[f])
        print(f"{t.dataset_name} fold{f}: {n} predictions OK")
print("total:", sum(len(t.predictions[f]) for t in mb2.tasks for f in t.folds))

## Notes

- One model is fine-tuned per fold on the official train split (standard matbench protocol); all folds start from the same BigBang-Proton-MPTRJ weights.
- Full training/inference code lives in the BigBang-Proton materials repository; this notebook provides the matbench integration.
- Requirements: python>=3.8, numpy, matbench>=0.6. See `info.json` for details.